# E15 · Ensemble of the best models

No training. Several strong models each predict every Kaggle image (8 rotated/flipped views), and their probabilities
are averaged. Members that are about equally good but disagree on the uncertain images can cancel each other's
mistakes: E13a, for example, changed 5% of E09a's answers, almost all on images where E09a was unsure.

The earlier ensemble (E07a: E06c + shape-focused model, 70.99%) did not help, but it combined a 71.50% model with a
clearly weaker one (70.24%). The members here are all within one point of the best model.

| Member | Run | Kaggle public |
|---|---|---|
| E09a | no AdaBN | 72.67% |
| E12a | + AdaBN | 72.39% |
| E13a | + AdaBN (spectral indices) | 72.11% |
| E11a | no AdaBN (two-branch) | 71.92% |
| E14a | no AdaBN (no B01, B09) | not yet scored |

Every member is loaded as it was submitted (an AdaBN run folder already holds the adapted weights), with its own bands
and input layer from its `config.json`, so 10-band (E14a) and index models (E13a) can be mixed.

**Variants:** `ENSEMBLES` lists which members each variant averages. `LOAD_ENSEMBLES` points a variant to its saved
folder; `None` creates a new one. Nothing is trained, the notebook runs in a few minutes.

## 1 · Setup and data

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.data import CLASSES, PROJECT_ROOT, TEST_BANDS
from src.experiment import class_mix, find_run, kaggle_score, load_experiment_data, plot_class_mix, view_agreement
from src.plot_style import apply_style
from src.training import load_run, new_run_dir, predict_views, write_submission
from src.utils import describe_device, get_device

apply_style()

# ---- Experiment settings ---------------------------------------------------------------
NOTEBOOK = "E15_ensemble"
MEMBERS = {   # short name -> run folder name without the date prefix
    "E09a": "pl_r3_view_agree_contrast_aug_hist_split_effb0",
    "E12a+AdaBN": "adabn_balanced_pl_r4_view_agree_contrast_aug_hist_split_effb0",
    "E13a+AdaBN": "adabn_balanced_indices_pl_r3_view_agree_contrast_aug_hist_split_effb0",
    "E17a": "l2a_pl_r3_view_agree_contrast_aug_effb0",
    "E17b": "l2a_summer_pl_r3_view_agree_contrast_aug_effb0",
}
ENSEMBLES = {   # variant -> members averaged with equal weight
    # done earlier: E15a (E09a, E12a+AdaBN, E13a+AdaBN) 72.81%, E15b (+ E11a) 72.63%, E15c (+ E11a, E14a) 72.58%,
    #               E15d (E15a + E17a) 72.95%, E15e (E09a, E13a+AdaBN, E17a) 72.77%
    "E15f": ["E09a", "E12a+AdaBN", "E13a+AdaBN", "E17a", "E17b"],
    "E15g": ["E09a", "E12a+AdaBN", "E13a+AdaBN", "E17b"],
}
LOAD_ENSEMBLES = {   # saved ensemble folders; None = create a new folder
    "E15f": None,
    "E15g": None,
}
REFERENCE = "E09a"   # best single model, the one to beat
# -----------------------------------------------------------------------------------------

device = get_device()
print("Device:", describe_device(device))
data = load_experiment_data(list(TEST_BANDS), ["hist_split"], device)   # all 12 bands; members pick theirs
print(data.summary())

## 2 · Member predictions

Each member predicts the Kaggle images (8 views) and the internal EuroSAT test split (hist_split-corrected). The
internal test of an AdaBN member is only indicative: its BatchNorm statistics were adapted to Kaggle.

In [ ]:
views, te_prob, dirs = {}, {}, {}
for name, run in MEMBERS.items():
    dirs[name] = find_run(run)
    model, normalizer, config = load_run(dirs[name], device)
    idx = torch.tensor([TEST_BANDS.index(b) for b in config["bands"]], device=device)
    views[name] = predict_views(model, data.kg_images[:, idx], normalizer)
    te_prob[name] = predict_views(model, data.te_images[:, idx], normalizer).mean(0)
    print(f"{name:12s} {len(config['bands'])} bands  {dirs[name].name}")
    del model
    if device.type == "mps":
        torch.mps.empty_cache()

prob = {k: v.mean(0) for k, v in views.items()}
ref_pred = prob[REFERENCE].argmax(1)
members = pd.DataFrame({
    "Kaggle public (%)": {k: kaggle_score(d) for k, d in dirs.items()},
    "internal test (%)": {k: (p.argmax(1) == data.y_te).mean() * 100 for k, p in te_prob.items()},
    "confidence (%)": {k: p.max(1).mean() * 100 for k, p in prob.items()},
    "view agreement (%)": {k: view_agreement(v) * 100 for k, v in views.items()},
    f"same class as {REFERENCE} (%)": {k: (p.argmax(1) == ref_pred).mean() * 100 for k, p in prob.items()},
})
members.round(2)

In [ ]:
# How often two members give the same class (%). Lower = more diverse = more to gain from averaging.
names = list(prob)
pd.DataFrame([[(prob[a].argmax(1) == prob[b].argmax(1)).mean() * 100 for b in names] for a in names],
             index=names, columns=names).round(1)

## 3 · Ensembles

In [ ]:
summary = {}
ens_prob = {}
for variant, chosen in ENSEMBLES.items():
    v = np.mean([views[m] for m in chosen], axis=0)        # (8, N, 10)
    p = v.mean(0)
    ens_prob[variant] = p
    if LOAD_ENSEMBLES[variant]:
        ens_dir = PROJECT_ROOT / LOAD_ENSEMBLES[variant]
    else:
        ens_dir = new_run_dir(f"ensemble_{variant.lower()}_{'_'.join(m.split('+')[0].lower() for m in chosen)}")
        (ens_dir / "config.json").write_text(json.dumps({
            "run_name": ens_dir.name, "notebook": NOTEBOOK, "variant": variant,
            "method": "ensemble (mean of TTA probabilities)",
            "description": "average of " + ", ".join(chosen),
            "members": [str(dirs[m].relative_to(PROJECT_ROOT)) for m in chosen]}, indent=2))
        te = np.mean([te_prob[m] for m in chosen], axis=0)
        (ens_dir / "metrics.json").write_text(json.dumps({
            "internal_test_acc": float((te.argmax(1) == data.y_te).mean()),
            "kaggle_mean_confidence": float(p.max(1).mean()), "kaggle_view_agreement": view_agreement(v),
            "kaggle_predicted_mix_pct": class_mix(p).round(2).to_dict(), "kaggle_public_score": None}, indent=2))
    write_submission(p, data.test_ids, ens_dir / "submission.csv")
    summary[variant] = {"members": ", ".join(chosen), "Kaggle public (%)": kaggle_score(ens_dir),
                        "confidence (%)": p.max(1).mean() * 100, "view agreement (%)": view_agreement(v) * 100,
                        f"same class as {REFERENCE} (%)": (p.argmax(1) == ref_pred).mean() * 100,
                        "folder": ens_dir.name}
    print(f"{variant}: {ens_dir.relative_to(PROJECT_ROOT)}/submission.csv")
pd.DataFrame(summary).T

In [ ]:
# Where each ensemble changes E09a's answers: E09a's class (rows) -> ensemble's class (columns)
ref_names = np.array(CLASSES)[ref_pred]
for variant, p in ens_prob.items():
    new = np.array(CLASSES)[p.argmax(1)]
    changed = new != ref_names
    print(f"{variant}: {changed.sum()} images differ from {REFERENCE} ({changed.mean():.1%})")
    display(pd.crosstab(ref_names[changed], new[changed]).rename_axis(index=REFERENCE, columns=variant))

In [ ]:
plot_class_mix(pd.DataFrame({"train split": class_mix(np.eye(10)[data.y_tr]), REFERENCE: class_mix(prob[REFERENCE]),
                             **{k: class_mix(p) for k, p in ens_prob.items()}}))

## 4 · Conclusion

*To fill in after the Kaggle submission.* Write each score into the ensemble folder's `metrics.json`.

| Variant | Members | Kaggle public |
|---|---|---|
| E09a (best single model) | – | **72.67%** |
| E15a | E09a, E12a + AdaBN, E13a + AdaBN | |
| E15b | E15a + E11a | |
| E15c | E15b + E14a | |
| E15d | E15a + E17a | 72.95% |
| E15e | E09a, E13a + AdaBN, E17a | 72.77% |
| E15f | E15d + E17b | |
| E15g | E15a + E17b | |